In [1]:
import os

In [2]:
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/jdk-17.jdk/Contents/Home"
os.environ['SPARK_HOME'] = "/Users/jyotisingh/Downloads/spark-3.5.9-bin-hadoop3"
os.environ['PYSPARK_PYTHON'] = 'python'
os.environ['PYSPARK_DRIVER_PYTHON'] = 'jupyter'

In [3]:
from pyspark.sql import SparkSession

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, count, expr, current_date, current_timestamp, col, unix_timestamp
from pyspark.sql.functions import broadcast

In [5]:
from pyspark.sql.functions import col, row_number, rank, dense_rank, lag, lead, sum, avg, ntile, percent_rank, when
from pyspark.sql.window import Window

Build an end-to-end ETL pipeline using PySpark — all within a Jupyter Notebook environment. 

You’ll extract data (CSV and Parquet), transform it using Spark DataFrame operations, and load it into optimized storage with partitioning and bucketing strategies. 

Along the way, you’ll apply best practices for schema handling, null value treatment, deduplication, filtering, joins, aggregations, and incremental updates using timestamp filtering. 

This notebook focuses on practical techniques for building performant ETL pipelines that are easy to run, test, and maintain. 

You’ll be able to extract data from multiple formats, perform scalable transformations using PySpark, and write out optimized datasets ready for downstream analytics. 

To read a Parquet file (yellow_tripdata_2026-01.parquet) in PySpark, use the **spark.read.parquet() method**, which loads the data directly into a DataFrame.

In [6]:
spark = SparkSession.builder.master("local").appName("nyc_trips_analysis_etl").getOrCreate()

### Step 1: Extracting Data from Multiple Sources

Efficiently extracting this data is a crucial first step in building a robust Extract, Transform, and Load (ETL) pipeline.

In [7]:
# Read a Parquet file

trips = spark.read.parquet("/Users/jyotisingh/Downloads/yellow_tripdata_2026-01.parquet")

In [8]:
# Read a csv file

taxi_zones = spark.read.csv("/Users/jyotisingh/Downloads/taxi_zone_lookup.csv",header=True)

Inspect the contents

In [9]:
# Inspect the schema

trips.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [10]:
# total records
total_records = trips.count() #3724889

In [11]:
total_records

3724889

In [12]:
# Preview the trips data

trips.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2026-01-01 00:54:04|  2026-01-01 00:59:37|              1|         0.97|         1|                 N|         239|    

In [14]:
# Preview the trip zones data

taxi_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows



In [15]:
taxi_zones.columns

['LocationID', 'Borough', 'Zone', 'service_zone']

In [16]:
taxi_zones_renamed = taxi_zones

for col_name in taxi_zones.columns:
    taxi_zones_renamed = taxi_zones_renamed.withColumnRenamed(col_name, f"{col_name}_PU")

In [17]:
taxi_zones_renamed.columns

['LocationID_PU', 'Borough_PU', 'Zone_PU', 'service_zone_PU']

In [18]:
taxi_zones_filtered = taxi_zones_renamed.drop('service_zone_PU')

### Step 2: Transforming and cleaning Data
    
Clean, transform, and optimize data using PySpark. 
These processes ensure data integrity, consistency, and reliability for downstream analytics.


Data transformation is a key component of ETL pipelines, ensuring raw data is standardized, deduplicated, enriched, and structured properly. 

In this step, you will perform missing value handling, deduplication, type conversions, joins, aggregations, and custom transformations.

In [19]:
# Drop columns not required for this analysis

columns_to_drop = ['store_and_fwd_flag','RatecodeID','extra', 'mta_tax', 'tolls_amount','improvement_surcharge', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']

In [20]:
trips_subset = trips.drop(*columns_to_drop)

In [21]:
trips_subset.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'tip_amount',
 'total_amount']

#### Handling Missing Values

- Count the number of nulls for each column.
- Drop records with excessive missing values.

In [22]:
trips_subset.select( [sum(col(c).isNull().cast("int")).alias(c) for c in trips_subset.columns] ).show()

+--------+--------------------+---------------------+---------------+-------------+------------+------------+------------+-----------+----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|PULocationID|DOLocationID|payment_type|fare_amount|tip_amount|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------+------------+------------+-----------+----------+------------+
|       0|                   0|                    0|        1088058|            0|           0|           0|           0|          0|         0|           0|
+--------+--------------------+---------------------+---------------+-------------+------------+------------+------------+-----------+----------+------------+



In [25]:
#Inspecting a few records with NULL values

trips_subset.filter(col("passenger_count").isNull()).show(5)

+--------+--------------------+---------------------+---------------+-------------+------------+------------+------------+-----------+----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|PULocationID|DOLocationID|payment_type|fare_amount|tip_amount|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------+------------+------------+-----------+----------+------------+
|       2| 2026-01-01 00:28:47|  2026-01-01 00:49:30|           NULL|        10.58|          43|         200|           0|      82.01|       0.0|       89.19|
|       2| 2026-01-01 00:54:24|  2026-01-01 01:15:01|           NULL|        11.04|         200|          51|           0|      51.77|       0.0|       53.27|
|       2| 2026-01-01 00:12:31|  2026-01-01 00:41:02|           NULL|         4.25|         142|         234|           0|      32.17|       0.0|       36.92|
|       2| 2026-01-01 00:48:56|  2026-01-01 01

#### Find anomalies

- Find records where trip_distance == 0 but fare_amount > 0 to identify data cleaning needs.

In [26]:
trips_subset.filter((col("trip_distance") == 0) | (col("fare_amount") == 0) ).count()

127319

In [27]:
# Filter valid records: keeping trips with trip distance > 0.

trips_subset_clean = trips_subset.filter((col("trip_distance") > 0) )

In [33]:
# Filter valid records: keeping trips with fare amount > 0.

trips_filtered = trips_subset_clean.filter((col("fare_amount") > 0) ) 

In [34]:
trips_filtered.count()

3560862

#### Join Datasets

- Join trips_filtered with taxi_zones_filtered to enrich trip records.

In [35]:
result_df = trips_filtered.join(broadcast(taxi_zones_filtered),trips_filtered["PULocationID"]==taxi_zones_filtered["LocationID_PU"], how="left").drop("LocationID_PU")

In [36]:
result_df.show()

+--------+--------------------+---------------------+---------------+-------------+------------+------------+------------+-----------+----------+------------+----------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|PULocationID|DOLocationID|payment_type|fare_amount|tip_amount|total_amount|Borough_PU|             Zone_PU|
+--------+--------------------+---------------------+---------------+-------------+------------+------------+------------+-----------+----------+------------+----------+--------------------+
|       2| 2026-01-01 00:54:04|  2026-01-01 00:59:37|              1|         0.97|         239|         238|           1|        7.2|      3.66|       15.86| Manhattan|Upper West Side S...|
|       1| 2026-01-01 00:34:04|  2026-01-01 00:39:47|              0|          0.9|         163|         162|           2|        7.9|       0.0|       13.65| Manhattan|       Midtown North|
|       1| 2026-01-01 00:57:06|  2026-01-01 0

#### Aggregation

- Create aggregated data views such as daily trips and hourly trips for downstream analysis.

In [37]:
# Generate daily_trip files.

daily_trips = trips_filtered.withColumn("pickup_date",F.to_date("tpep_pickup_datetime")) \
                            .groupBy("pickup_date") \
                            .agg(
                                F.count("*").alias("total_rides"),
                                F.round((F.sum(col("tip_amount")))).cast("int").alias("total_tip_amount"),
                                F.round((F.sum(col("fare_amount")))).cast("int").alias("total_fare_amount"),
                                F.round((F.sum(col("total_amount")))).cast("int").alias("total_amount")
                                )\
                            .orderBy("pickup_date")

In [38]:
daily_trips.show()

+-----------+-----------+----------------+-----------------+------------+
|pickup_date|total_rides|total_tip_amount|total_fare_amount|total_amount|
+-----------+-----------+----------------+-----------------+------------+
| 2025-12-31|          5|               4|              276|         330|
| 2026-01-01|     108175|          250869|          2588518|     3488550|
| 2026-01-02|      94445|          268608|          2010155|     2856203|
| 2026-01-03|     105858|          285422|          2211793|     3089791|
| 2026-01-04|      91113|          254075|          2004033|     2782701|
| 2026-01-05|      92464|          275719|          2003578|     2838306|
| 2026-01-06|     102840|          293674|          2091036|     2997450|
| 2026-01-07|     107900|          308816|          2161961|     3113708|
| 2026-01-08|     114275|          336006|          2327580|     3349736|
| 2026-01-09|     117774|          331403|          2359403|     3398382|
| 2026-01-10|     142523|          338

In [39]:
# Generate hourly_trip files.

hourly_trips = trips_filtered.withColumn("pickup_date",F.to_date("tpep_pickup_datetime")) \
                            .withColumn("pickup_hour",F.hour("tpep_pickup_datetime")) \
                            .groupBy("pickup_date", "pickup_hour") \
                            .agg(
                                F.count("*").alias("total_rides"),
                                F.round((F.sum(col("tip_amount")))).cast("int").alias("total_tip_amount"), 
                                F.round((F.sum(col("fare_amount")))).cast("int").alias("total_fare_amount"),
                                F.round((F.sum(col("total_amount")))).cast("int").alias("total_amount")
                                )\
                            .orderBy("pickup_date","pickup_hour")

In [40]:
hourly_trips.show()

+-----------+-----------+-----------+----------------+-----------------+------------+
|pickup_date|pickup_hour|total_rides|total_tip_amount|total_fare_amount|total_amount|
+-----------+-----------+-----------+----------------+-----------------+------------+
| 2025-12-31|         23|          5|               4|              276|         330|
| 2026-01-01|          0|       9112|           19355|           220431|      293450|
| 2026-01-01|          1|      10059|           18504|           315221|      393472|
| 2026-01-01|          2|       7629|           15062|           207103|      265424|
| 2026-01-01|          3|       6372|            9711|           162315|      207809|
| 2026-01-01|          4|       3986|            5368|            94858|      122457|
| 2026-01-01|          5|       2137|            2970|            53514|       67594|
| 2026-01-01|          6|       2039|            2695|            62475|       75913|
| 2026-01-01|          7|       1884|            3413|

### Step 3: Loading Data Efficiently

Once data is extracted and transformed, the next step is to store it efficiently for downstream analytics and reporting. Efficient data storage improves:

Query Performance – Reduces scan time and improves retrieval speed
Scalability – Helps manage large datasets effectively
Storage Optimization – Reduces redundancy and improves data compression

In this step, you will optimize how data is stored in PySpark by:

Writing data efficiently to Parquet format/csv format.

Write Data to Parquet

- Parquet is a columnar storage format optimized for analytics. Writing data in Parquet format:

Reduces file size due to built-in compression
Speeds up query execution for analytical workloads

Save a sample of the daily_trips DataFrame in Parquet format to local disk. This helps you test the write process without overloading your system.


**Note:** 

To write a PySpark DataFrame to a CSV with a custom name, you cannot do it directly within Spark because Spark's distributed architecture natively outputs partitioned part-files like part-00000...csv. 

To achieve a custom file name, you must output the file using a single partition and then use an underlying file system tool to rename it.

In [41]:
daily_trips.write.mode("overwrite").parquet("/Users/jyotisingh/Downloads/yellow_tripdata/Daily_trips/daily_trips_2026_01.parquet")

In [42]:
# Standard way to write a PySpark DataFrame to a CSV folder

daily_trips.write.format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .option("delimiter", ",") \
   .save("/Users/jyotisingh/Downloads/yellow_tripdata/Daily_trips")

In [43]:
spark.stop()